In [1]:
%run ../module7-genai-langchain/initialize_environment.py

Environment initializing completed successfully.


In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = create_azure_llm("chat")
large_model.name = "Large Context Model"

standard_model = create_azure_llm("chat")
standard_model.name = "Standard Model"



@wrap_model_call #Marks this as a LangChain middleware that intercepts and modifies model requests/responses
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    # Override the model for this request
    request = request.override(model=model)  

    #this modified request is then passed to the next handler in the chain, which will ultimately call the selected model
    
    return handler(request)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview
Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=large_model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

Not yet! I can take care of that right now. Would you like me to check if it needs any other care too?


In [ ]:
print(response["messages"][-1].response_metadata)

In [7]:
print(response["messages"][-1].response_metadata['system_fingerprint'])

fp_b6f445fc1c


In [8]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

Usually, we’ll need to repot the plant every 12 to 18 months or if you notice roots growing out of the drainage holes. Since it’s still healthy and growing steadily, we probably won’t need to repot it for a few more months. I’ll keep an eye on it and let you know when it’s time!


In [10]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-4.1-mini-2025-04-14


In [9]:
print(response["messages"][-1].response_metadata['system_fingerprint'])

fp_b6f445fc1c
